In [1]:
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output, State
import plotly.graph_objects as go
import plotly.express as px
from dash import no_update
import datetime as dt
import requests

In [2]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv'

def download(filename, url):
    response = requests.get(url)
    if response.status_code == 200:
        with open (filename, 'wb') as f:
            f.write(response.content)

download('spacex_dash.csv', URL)

In [3]:
df = pd.read_csv('spacex_dash.csv')
df.head(5)

,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [4]:
df['Launch Site'].value_counts()

Launch Site
CCAFS LC-40     26
KSC LC-39A      13
VAFB SLC-4E     10
CCAFS SLC-40     7
Name: count, dtype: int64

## Use the given Skeleton for the App

In [17]:
# Read the airline data into pandas dataframe
spacex_df = df
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

# Create a dash application
app = dash.Dash(__name__)

# Create an app layout
app.layout = html.Div(children=[html.H1('SpaceX Launch Records Dashboard',
                                        style={'textAlign': 'center', 'color': '#503D36',
                                               'font-size': 40}),
                                # TASK 1: Add a dropdown list to enable Launch Site selection
                                # The default select value is for ALL sites
                                dcc.Dropdown(id='site-dropdown',
                                            options = [
                                                {'label' : 'All Sites', 'value' : 'ALL'},
                                                {'label' : 'CCAFS LC-40', 'value' : 'CCAFS LC-40'},
                                                {'label' : 'KSC LC-39A', 'value' : 'KSC LC-39A'},
                                                {'label' : 'VAFB SLC-4E', 'value' : 'VAFB SLC-4E'},
                                                {'label' : 'CCAFS SLC-40', 'value' : 'CCAFS SLC-40'}
                                            ],
                                            value = 'ALL',
                                            placeholder = 'Select a Launch Site Here',
                                            searchable = True
                                            ),
                                html.Br(),

                                # TASK 2: Add a pie chart to show the total successful launches count for all sites
                                # If a specific launch site was selected, show the Success vs. Failed counts for the site
                                html.Div(dcc.Graph(id='success-pie-chart')),
                                html.Br(),

                                html.P("Payload range (Kg):"),
                                
                                # TASK 3: Add a slider to select payload range
                                dcc.RangeSlider(
                                    id='payload-slider',
                                    min = 0,
                                    max = 10000,
                                    step = 1000,
                                    marks = {0: '0',
                                            100: '100'},
                                    value = [min_payload, max_payload]
                                ),

                                # TASK 4: Add a scatter chart to show the correlation between payload and launch success
                                html.Div(dcc.Graph(id='success-payload-scatter-chart')),
                                ])

# TASK 2:
# Add a callback function for `site-dropdown` as input, `success-pie-chart` as output
@app.callback(
    Output(component_id = 'success-pie-chart', component_property = 'figure'),
    Input(component_id = 'site-dropdown', component_property = 'value')
)

def get_pie_chart(entered_site):
    if entered_site == 'ALL':
        fig = px.pie(
            spacex_df,
            values = 'class',
            names = 'Launch Site',
            title = 'Succesfull Launches in ALL Launch Sites'
        )
    else:
        filtered_df = spacex_df[spacex_df['Launch Site'] == entered_site]
        df_counts = filtered_df['class'].value_counts().reset_index()
        df_counts.columns = ['class', 'count']
        fig = px.pie(
            df_counts,
            values = 'count',
            names = 'class',
            title = f'Successful Launches in Site: {entered_site}'
        )
    return fig
# TASK 4:
# Add a callback function for `site-dropdown` and `payload-slider` as inputs, `success-payload-scatter-chart` as output
@app.callback(
    Output(component_id = 'success-payload-scatter-chart', component_property = 'figure'),
    [Input(component_id = 'site-dropdown', component_property = 'value'),
    Input(component_id = 'payload-slider', component_property = 'value')]
)

def get_scatter_chart(entered_site, entered_payload):

    min, max = entered_payload
    filt = (spacex_df['Payload Mass (kg)'] >= min) & (spacex_df['Payload Mass (kg)'] <= max)
    filt_df = spacex_df[filt]
    if entered_site == 'ALL':
        fig = px.scatter(
            filt_df,
            x = 'Payload Mass (kg)',
            y = 'class',
            color = 'Booster Version Category'
        )
    else:
        df_site = filt_df[filt_df['Launch Site'] == entered_site]
        fig = px.scatter(
            df_site,
            x = 'Payload Mass (kg)',
            y = 'class',
            color = 'Booster Version Category'
        )
    return fig

# Run the app
if __name__ == '__main__':
    app.run()

